# Análise de Falhas de Equipamentos Industriais — Versão Final Enxuta

Este notebook consolida a análise da base de falhas fornecida no processo seletivo, com foco em qualidade dos dados, revisão de `MODO_FALHA` e `IMPACTO_FALHA`, tratamento de campos ausentes, identificação de duplicidades e geração da planilha final padronizada.

A abordagem final combina **regras semânticas de alta confiança** com **modelos de classificação textual como fallback**, preservando rastreabilidade e evitando correções automáticas quando não há evidência suficiente.

## 1. Bibliotecas e configuração

In [1]:
from pathlib import Path
import base64
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pd.set_option("display.max_colwidth", 120)

candidatos = [
    Path("/content/Falhas_Processo_Seletivo.xlsx"),
    Path("Falhas_Processo_Seletivo.xlsx"),
    Path("/mnt/data/Falhas_Processo_Seletivo.xlsx")
]

arquivo = next((p for p in candidatos if p.exists()), None)
if arquivo is None:
    raise FileNotFoundError("Envie Falhas_Processo_Seletivo.xlsx para o Colab antes de executar o notebook.")

pasta_saida = Path("/content") if Path("/content").exists() else Path.cwd()
print("Arquivo encontrado:", arquivo)
print("Pasta de saída:", pasta_saida)

Arquivo encontrado: Falhas_Processo_Seletivo.xlsx
Pasta de saída: /mnt/data


## 2. Carregamento e auditoria inicial

In [2]:
df = pd.read_excel(arquivo, sheet_name="FALHAS")

resumo_ausentes = pd.DataFrame({
    "Quantidade_Ausente": df.isna().sum(),
    "Percentual_Ausente": (df.isna().mean() * 100).round(2)
})

print("Dimensão da base:", df.shape)
display(resumo_ausentes)
display(df.head())

Dimensão da base: (10000, 7)


,Quantidade_Ausente,Percentual_Ausente
ID,0,0.00
DATA_FALHA,37,0.37
SISTEMA,0,0.00
TAG,0,0.00
DESCRICAO_FALHA,0,0.00
MODO_FALHA,1687,16.87
IMPACTO_FALHA,1592,15.92


,ID,DATA_FALHA,SISTEMA,TAG,DESCRICAO_FALHA,MODO_FALHA,IMPACTO_FALHA
0,1,2019-03-26 00:00:00,Sistema Elétrico Principal,QD-13018B,Painel elétrico QD-13018B n/ partiu o equip. associado.,FTS,"Degradação severa, equipamento funcional"
1,2,2024-07-13 00:00:00,Sistema de Trocadores de Calor,E-11024B,Trocador E-11024B não respondeu à partida do sistema de circulação.,NaN,Falha: perda da função requerida
2,3,2020-09-13 00:00:00,Sistema de Armazenamento e Tanques,TQ-10010,Sistema de drenagem do tanque TQ-10010 não partiu sob demanda.,FTS,Falha: perda da função requerida
3,4,2023-10-19 00:00:00,Sistema Elétrico Principal,QD-13010,Perda de função do painel QD-13010 em operação.,FTF,Falha: perda da função requerida
4,5,2026-07-17 00:00:00,Sistema de Compressão de Gás,K-3018B,Comrpessor K-3018B apresentuo trips consecutivos durante o turno.,FTF,Equipamento funcional


### Síntese da qualidade inicial

A auditoria verifica completude, estrutura, datas inválidas e duplicidades lógicas. A base original é mantida inalterada durante toda a análise.

## 3. Validação de datas

In [3]:
datas_convertidas = pd.to_datetime(df["DATA_FALHA"], errors="coerce", dayfirst=True)

datas_invalidas = df[
    df["DATA_FALHA"].notna() & datas_convertidas.isna()
][["ID", "DATA_FALHA"]]

print("Datas ausentes originalmente:", df["DATA_FALHA"].isna().sum())
print("Datas inválidas preenchidas:", len(datas_invalidas))
print("Total problemático após conversão:", datas_convertidas.isna().sum())
display(datas_invalidas["DATA_FALHA"].value_counts().to_frame("Quantidade"))

Datas ausentes originalmente: 37
Datas inválidas preenchidas: 155
Total problemático após conversão: 192


,Quantidade
DATA_FALHA,
N/D,45
0000-00-00,40
--,38
erro,32


## 4. Identificação de duplicidades lógicas

In [4]:
chaves_duplicidade = ["DATA_FALHA", "SISTEMA", "TAG", "DESCRICAO_FALHA"]

mask_duplicado = df.duplicated(subset=chaves_duplicidade, keep=False)
duplicados = df.loc[mask_duplicado].copy()

grupos_duplicados = (
    df.groupby(chaves_duplicidade, dropna=False)
      .size()
      .reset_index(name="Quantidade")
      .query("Quantidade > 1")
)

print("Registros envolvidos em duplicidade:", int(mask_duplicado.sum()))
print("Grupos duplicados:", len(grupos_duplicados))
display(grupos_duplicados["Quantidade"].value_counts().sort_index().to_frame("Grupos"))

Registros envolvidos em duplicidade: 882
Grupos duplicados: 441


,Grupos
Quantidade,
2,441


### Conclusão da auditoria inicial

A análise evidencia problemas reais de qualidade: campos ausentes, datas inválidas e registros duplicados. As duplicidades são avaliadas pelo evento descrito, e não pelo `ID`, pois cada linha possui identificador próprio.

## 5. Regras semânticas finais para `MODO_FALHA`

In [5]:
def classificar_modo(descricao):
    texto = str(descricao).lower()

    if any(t in texto for t in ["vibração", "vibracao", "vibra", "vib.", "rms"]):
        return "VIB"
    if any(t in texto for t in ["ruído", "ruido", "barulho"]):
        return "NOI"
    if any(t in texto for t in [
        "trinca", "fissura", "corros", "deforma",
        "desgaste estrutural", "degradação estrutural", "degradacao estrutural",
        "deterioração estrutural", "deterioracao estrutural",
        "degradação da fundação", "degradacao da fundacao",
        "estrutura comprometida", "fixação solta", "fixacao solta"
    ]):
        return "STD"
    if any(t in texto for t in [
        "falha de partida", "não partiu", "nao partiu", "n/ partiu",
        "não deu partida", "nao deu partida",
        "não respondeu à partida", "nao respondeu a partida", "n/ respondeu à partida",
        "não iniciou", "nao iniciou", "n/ iniciou", "nao inicoui",
        "não energizou", "nao energizou",
        "não entrou em operação", "nao entrou em operacao",
        "não entrou em op", "nao entrou em op",
        "não entrou em funcionamento", "nao entrou em funcionamento"
    ]):
        return "FTS"
    if any(t in texto for t in [
        "não acompanhou", "nao acompanhou",
        "perda de função", "perda de funcao",
        "perda de vazão", "perda de vazao",
        "perda de troca térmica", "perda de troca termica",
        "perda de capacidade", "parou", "desligou", "desarmou", "trip",
        "não atuou", "nao atuou",
        "não respondeu ao comando", "nao respondeu ao comando", "n/ respondeu ao comando",
        "não movimentou", "nao movimentou",
        "perdeu vedação", "perdeu vedacao", "ficou travada",
        "não recompôs", "nao recompos", "n/ recompôs",
        "operou com vazão abaixo", "operou com vazao abaixo",
        "não fechou", "nao fechou",
        "não permaneceu fechada", "nao permaneceu fechada"
    ]):
        return "FTF"
    return "AMBIGUO"


## 6. Regra conservadora para `IMPACTO_FALHA`

In [6]:
def classificar_impacto_alta_confianca(descricao):
    texto = str(descricao).lower()

    termos_falha = [
        "não partiu", "nao partiu", "n/ partiu", "não deu partida", "nao deu partida",
        "não respondeu à partida", "nao respondeu a partida", "não iniciou", "nao iniciou", "n/ iniciou",
        "não entrou em operação", "nao entrou em operacao", "não atuou", "nao atuou", "n/ atuou",
        "não respondeu ao comando", "nao respondeu ao comando", "não movimentou", "nao movimentou",
        "não abriu", "nao abriu", "não fechou", "nao fechou", "ficou travada", "permaneceu bloqueada",
        "parou", "desligou", "desarmou", "desamrou", "trip", "trips", "perda de função", "perda de funcao"
    ]

    termos_degradacao = [
        "vazão abaixo", "vazao abaixo", "perda de capacidade", "degradação", "degradacao",
        "desgaste estrutural", "corrosão", "corrosao", "trinca", "fissura", "deformação", "deformacao",
        "vazamento", "histerese", "danos na vedação", "danos na vedacao"
    ]

    if any(t in texto for t in termos_falha):
        return "Falha: perda da função requerida"
    if any(t in texto for t in termos_degradacao):
        return "Degradação severa, equipamento funcional"
    return "AMBIGUO"


## 7. Aplicação das regras e análise de divergências

In [7]:
df_trabalho = df.copy()
df_trabalho["MODO_REGRA"] = df_trabalho["DESCRICAO_FALHA"].apply(classificar_modo)
df_trabalho["IMPACTO_ALTA_CONFIANCA"] = df_trabalho["DESCRICAO_FALHA"].apply(classificar_impacto_alta_confianca)

divergencias_modo = df_trabalho[
    df_trabalho["MODO_FALHA"].notna() &
    (df_trabalho["MODO_REGRA"] != "AMBIGUO") &
    (df_trabalho["MODO_FALHA"] != df_trabalho["MODO_REGRA"])
]

divergencias_impacto = df_trabalho[
    df_trabalho["IMPACTO_FALHA"].notna() &
    (df_trabalho["IMPACTO_ALTA_CONFIANCA"] != "AMBIGUO") &
    (df_trabalho["IMPACTO_FALHA"] != df_trabalho["IMPACTO_ALTA_CONFIANCA"])
]

print("Divergências candidatas em MODO_FALHA:", len(divergencias_modo))
print("Divergências candidatas em IMPACTO_FALHA:", len(divergencias_impacto))

Divergências candidatas em MODO_FALHA: 1199
Divergências candidatas em IMPACTO_FALHA: 1459


### Critério de decisão

As divergências não são tratadas automaticamente como erros apenas por discordarem do dado original. Para `MODO_FALHA`, as regras foram validadas sobre amostras de descrições com evidência textual clara. Para `IMPACTO_FALHA`, a correção é mais conservadora e só ocorre quando há evidência explícita de perda funcional ou degradação severa.

## 8. Modelos de fallback para casos não resolvidos

In [8]:
treino_modo = df_trabalho[
    (df_trabalho["MODO_REGRA"] != "AMBIGUO") &
    (df_trabalho["MODO_FALHA"].notna()) &
    (df_trabalho["MODO_FALHA"] == df_trabalho["MODO_REGRA"])
].copy()

modelo_modo = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])
modelo_modo.fit(treino_modo["DESCRICAO_FALHA"], treino_modo["MODO_FALHA"])

treino_impacto_explicito = df_trabalho[
    (df_trabalho["IMPACTO_ALTA_CONFIANCA"] != "AMBIGUO") &
    (df_trabalho["IMPACTO_FALHA"].notna()) &
    (df_trabalho["IMPACTO_FALHA"] == df_trabalho["IMPACTO_ALTA_CONFIANCA"])
].copy()

treino_impacto_funcional = df_trabalho[
    (df_trabalho["IMPACTO_FALHA"] == "Equipamento funcional") &
    (df_trabalho["MODO_REGRA"].isin(["VIB", "NOI"])) &
    (df_trabalho["MODO_FALHA"] == df_trabalho["MODO_REGRA"])
].copy()

treino_impacto = pd.concat([treino_impacto_explicito, treino_impacto_funcional], ignore_index=True).drop_duplicates(subset="ID")

modelo_impacto = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])
modelo_impacto.fit(treino_impacto["DESCRICAO_FALHA"], treino_impacto["IMPACTO_FALHA"])

print("Treino confiável — MODO_FALHA:", len(treino_modo))
print("Treino confiável — IMPACTO_FALHA:", len(treino_impacto))

Treino confiável — MODO_FALHA: 5306
Treino confiável — IMPACTO_FALHA: 3136


## 9. Consolidação final das classificações

In [9]:
df_trabalho["MODO_FINAL"] = df_trabalho["MODO_FALHA"]
mask_modo_regra = df_trabalho["MODO_REGRA"] != "AMBIGUO"
df_trabalho.loc[mask_modo_regra, "MODO_FINAL"] = df_trabalho.loc[mask_modo_regra, "MODO_REGRA"]
mask_modo_fallback = df_trabalho["MODO_FINAL"].isna()
df_trabalho.loc[mask_modo_fallback, "MODO_FINAL"] = modelo_modo.predict(df_trabalho.loc[mask_modo_fallback, "DESCRICAO_FALHA"])

df_trabalho["IMPACTO_FINAL"] = df_trabalho["IMPACTO_FALHA"]
mask_impacto_regra = df_trabalho["IMPACTO_ALTA_CONFIANCA"] != "AMBIGUO"
df_trabalho.loc[mask_impacto_regra, "IMPACTO_FINAL"] = df_trabalho.loc[mask_impacto_regra, "IMPACTO_ALTA_CONFIANCA"]
mask_impacto_fallback = df_trabalho["IMPACTO_FINAL"].isna()
df_trabalho.loc[mask_impacto_fallback, "IMPACTO_FINAL"] = modelo_impacto.predict(df_trabalho.loc[mask_impacto_fallback, "DESCRICAO_FALHA"])

display(df_trabalho["MODO_FINAL"].value_counts().to_frame("Quantidade"))
display(df_trabalho["IMPACTO_FINAL"].value_counts().to_frame("Quantidade"))

,Quantidade
MODO_FINAL,
VIB,3425
FTF,2216
STD,1879
NOI,1256
FTS,1224


,Quantidade
IMPACTO_FINAL,
"Degradação severa, equipamento funcional",3556
Equipamento funcional,3502
Falha: perda da função requerida,2942


## 10. Identificação do principal tipo de erro

In [10]:
mask_duplicado = df_trabalho.duplicated(subset=chaves_duplicidade, keep=False)
mask_campo_vazio = df_trabalho["MODO_FALHA"].isna() | df_trabalho["IMPACTO_FALHA"].isna()

mask_modo_incorreto = (
    df_trabalho["MODO_FALHA"].notna() &
    (df_trabalho["MODO_REGRA"] != "AMBIGUO") &
    (df_trabalho["MODO_FALHA"] != df_trabalho["MODO_REGRA"])
)
mask_impacto_incorreto = (
    df_trabalho["IMPACTO_FALHA"].notna() &
    (df_trabalho["IMPACTO_ALTA_CONFIANCA"] != "AMBIGUO") &
    (df_trabalho["IMPACTO_FALHA"] != df_trabalho["IMPACTO_ALTA_CONFIANCA"])
)
mask_classificacao_incorreta = mask_modo_incorreto | mask_impacto_incorreto
mask_descricao_ambigua = (
    (df_trabalho["MODO_REGRA"] == "AMBIGUO") &
    (df_trabalho["IMPACTO_ALTA_CONFIANCA"] == "AMBIGUO")
)

df_trabalho["TIPO_DE_ERRO"] = np.select(
    [mask_duplicado, mask_campo_vazio, mask_classificacao_incorreta, mask_descricao_ambigua],
    ["DUPLICADO", "CAMPO_VAZIO", "CLASSIFICACAO_INCORRETA", "DESCRICAO_AMBIGUA"],
    default="NENHUM"
)

df_trabalho["REGISTRO_CORRETO_OU_INCORRETO"] = np.where(
    df_trabalho["TIPO_DE_ERRO"] == "NENHUM", "CORRETO", "INCORRETO"
)

display(df_trabalho["TIPO_DE_ERRO"].value_counts().to_frame("Quantidade"))

,Quantidade
TIPO_DE_ERRO,
NENHUM,3588
CAMPO_VAZIO,2333
CLASSIFICACAO_INCORRETA,2045
DESCRICAO_AMBIGUA,1152
DUPLICADO,882


## 11. Construção da planilha de respostas

In [11]:
resposta_final = pd.DataFrame({
    "ID": df_trabalho["ID"],
    "MODO_FALHA": df_trabalho["MODO_FINAL"],
    "IMPACTO_FALHA": df_trabalho["IMPACTO_FINAL"],
    "REGISTRO_CORRETO_OU_INCORRETO": df_trabalho["REGISTRO_CORRETO_OU_INCORRETO"],
    "TIPO_DE_ERRO": df_trabalho["TIPO_DE_ERRO"]
})

display(resposta_final.head())

,ID,MODO_FALHA,IMPACTO_FALHA,REGISTRO_CORRETO_OU_INCORRETO,TIPO_DE_ERRO
0,1,FTS,Falha: perda da função requerida,INCORRETO,DUPLICADO
1,2,FTS,Falha: perda da função requerida,INCORRETO,CAMPO_VAZIO
2,3,FTS,Falha: perda da função requerida,CORRETO,NENHUM
3,4,FTF,Falha: perda da função requerida,CORRETO,NENHUM
4,5,FTF,Falha: perda da função requerida,INCORRETO,CLASSIFICACAO_INCORRETA


## 12. Validação final da entrega

In [12]:
modos_validos = {"FTS", "FTF", "STD", "VIB", "NOI"}
impactos_validos = {"Equipamento funcional", "Degradação severa, equipamento funcional", "Falha: perda da função requerida"}
status_validos = {"CORRETO", "INCORRETO"}
erros_validos = {"NENHUM", "CLASSIFICACAO_INCORRETA", "CAMPO_VAZIO", "DESCRICAO_AMBIGUA", "DUPLICADO"}

assert resposta_final.shape == (10000, 5)
assert resposta_final["ID"].is_unique
assert resposta_final.isna().sum().sum() == 0
assert set(resposta_final["MODO_FALHA"]).issubset(modos_validos)
assert set(resposta_final["IMPACTO_FALHA"]).issubset(impactos_validos)
assert set(resposta_final["REGISTRO_CORRETO_OU_INCORRETO"]).issubset(status_validos)
assert set(resposta_final["TIPO_DE_ERRO"]).issubset(erros_validos)

print("Validação concluída com sucesso.")
print("Linhas:", len(resposta_final))
print("IDs únicos:", resposta_final["ID"].nunique())
print("Valores ausentes:", resposta_final.isna().sum().sum())

Validação concluída com sucesso.
Linhas: 10000
IDs únicos: 10000
Valores ausentes: 0


## 13. Indicadores finais

In [13]:
indicadores = pd.Series({
    "Registros analisados": len(df_trabalho),
    "Registros corretos": (df_trabalho["REGISTRO_CORRETO_OU_INCORRETO"] == "CORRETO").sum(),
    "Registros incorretos": (df_trabalho["REGISTRO_CORRETO_OU_INCORRETO"] == "INCORRETO").sum(),
    "Duplicados": (df_trabalho["TIPO_DE_ERRO"] == "DUPLICADO").sum(),
    "Campos vazios": (df_trabalho["TIPO_DE_ERRO"] == "CAMPO_VAZIO").sum(),
    "Classificações incorretas": (df_trabalho["TIPO_DE_ERRO"] == "CLASSIFICACAO_INCORRETA").sum(),
    "Descrições ambíguas": (df_trabalho["TIPO_DE_ERRO"] == "DESCRICAO_AMBIGUA").sum()
})
display(indicadores.to_frame("Quantidade"))

,Quantidade
Registros analisados,10000
Registros corretos,3588
Registros incorretos,6412
Duplicados,882
Campos vazios,2333
Classificações incorretas,2045
Descrições ambíguas,1152


## 14. Exportação da planilha obrigatória

In [14]:
arquivo_resposta = pasta_saida / "Resposta_Candidato.xlsx"
resposta_final.to_excel(arquivo_resposta, sheet_name="RESPOSTAS", index=False)
print("Arquivo gerado:", arquivo_resposta)

Arquivo gerado: /mnt/data/Resposta_Candidato.xlsx


## 15. Visualização obrigatória em HTML autocontido

In [15]:
def figura_base64(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

fig1, ax1 = plt.subplots(figsize=(8, 4.5))
resposta_final["MODO_FALHA"].value_counts().plot(kind="bar", ax=ax1)
ax1.set_title("Distribuição Final dos Modos de Falha")
ax1.set_xlabel("Modo de Falha")
ax1.set_ylabel("Quantidade")
fig1.tight_layout()
img1 = figura_base64(fig1)

fig2, ax2 = plt.subplots(figsize=(9, 4.5))
resposta_final["TIPO_DE_ERRO"].value_counts().plot(kind="bar", ax=ax2)
ax2.set_title("Distribuição dos Tipos de Erro")
ax2.set_xlabel("Tipo de Erro")
ax2.set_ylabel("Quantidade")
ax2.tick_params(axis="x", rotation=30)
fig2.tight_layout()
img2 = figura_base64(fig2)

html = f"""<!doctype html>
<html lang='pt-BR'>
<head>
<meta charset='utf-8'>
<title>Visualização dos Resultados - Análise de Falhas</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 36px; color: #222; }}
h1 {{ margin-bottom: 4px; }}
.kpis {{ display: flex; gap: 16px; flex-wrap: wrap; margin: 24px 0; }}
.kpi {{ border: 1px solid #ddd; border-radius: 8px; padding: 14px 18px; min-width: 180px; }}
.kpi strong {{ display: block; font-size: 26px; margin-top: 6px; }}
.grafico {{ margin: 34px 0; }}
img {{ max-width: 900px; width: 100%; height: auto; }}
</style>
</head>
<body>
<h1>Análise de Falhas de Equipamentos Industriais</h1>
<p>Resumo visual dos resultados finais da análise.</p>
<div class='kpis'>
  <div class='kpi'>Registros analisados<strong>{len(resposta_final):,}</strong></div>
  <div class='kpi'>Registros incorretos<strong>{int((resposta_final['REGISTRO_CORRETO_OU_INCORRETO']=='INCORRETO').sum()):,}</strong></div>
  <div class='kpi'>Duplicados identificados<strong>{int((resposta_final['TIPO_DE_ERRO']=='DUPLICADO').sum()):,}</strong></div>
</div>
<div class='grafico'><img src='data:image/png;base64,{img1}'></div>
<div class='grafico'><img src='data:image/png;base64,{img2}'></div>
</body>
</html>"""

arquivo_html = pasta_saida / "Visualizacao_Resultados.html"
arquivo_html.write_text(html, encoding="utf-8")
print("Visualização gerada:", arquivo_html)

Visualização gerada: /mnt/data/Visualizacao_Resultados.html


## 16. Conclusão executiva

A análise dos 10.000 registros identificou problemas de completude, datas inválidas, duplicidades, descrições ambíguas e inconsistências entre as descrições textuais e as classificações registradas.

A solução final adota uma abordagem híbrida e auditável. Regras semânticas são priorizadas quando há evidência textual clara; modelos TF-IDF + Regressão Logística são utilizados apenas como fallback para completar casos não resolvidos diretamente pelas regras. Para `IMPACTO_FALHA`, a estratégia é deliberadamente mais conservadora, pois sintomas isolados nem sempre determinam a severidade funcional.

A planilha final preserva os 10.000 IDs originais, utiliza exclusivamente os domínios permitidos e informa, para cada registro, se ele foi considerado correto ou incorreto e qual foi o principal tipo de erro identificado.